# Method: Ridge Regression

#### Algorithm Reference

T. Hastie, R. Tibshirani, and J. Friedman, *The Elements of Statistical Learning: Data Mining, Inference, and Prediction*, 2nd ed. New York, NY, USA: Springer, 2009, sec. 3.4.1.

### I. Imports and Reads

#### Imports

In [27]:
import numpy as np
import csv 
import EvaluationMetrics as em

#### Reads

In [28]:
with open('../dataset/processed_test.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    test_in = np.array(data)
    test_out = np.array(output)

with open('../dataset/processed_tune.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    tune_in = np.array(data)
    tune_out = np.array(output)

with open('../dataset/processed_train.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    train_in = np.array(data)
    train_out = np.array(output)

train_in

array([['6', '0.01580336912619452', '-1.390444444444448',
        '38.190444444444445', '37.596296296296295', '37.00214814814815'],
       ['6', '0.01580336912619452', '-0.6904444444444451',
        '38.190444444444445', '37.596296296296295', '37.00214814814815'],
       ['6', '0.01580336912619452', '-2.190444444444445',
        '38.190444444444445', '37.596296296296295', '37.00214814814815'],
       ...,
       ['2', '0.02745569032449328', '0.5259574468084978',
        '37.2540425531915', '36.258539325842705', '36.4595294117647'],
       ['2', '0.02745569032449328', '-4.254042553191503',
        '37.2540425531915', '36.258539325842705', '36.4595294117647'],
       ['2', '0.02745569032449328', '0.07595744680849492',
        '37.2540425531915', '36.258539325842705', '36.4595294117647']],
      shape=(3109, 6), dtype='<U22')

#### Ridge Regression Functions

In [29]:
LAMB = 1  # hyperparameter, the lambda value

def compute_train_stats(X, y):
    x_mean = np.mean(X, axis=0) 			# one mean per column
    x_std = np.std(X, axis=0)				# one std per column
    y_mean = np.mean(y)
    return x_mean, x_std, y_mean

def standardize(X, x_mean, x_std):
    Xs = (X - x_mean) / x_std
    return Xs

def solve_coefficients(lamb, Xs, yc):
    p = Xs.shape[1]
    A = Xs.T @ Xs + lamb * np.eye(p)	  	# XTX + lambI
    b = Xs.T @ yc  							# XTy
    coeff = np.linalg.solve(A, b)
    return coeff

def fit(lamb, X, y):
    # get the training stats
    x_mean, x_std, y_mean = compute_train_stats(X, y)

	# standardize X and center y
    Xs = standardize(X, x_mean, x_std)
    yc = y - y_mean

	# solve for coeffs
    coeff = solve_coefficients(lamb, Xs, yc)
    intercept = y_mean

    return {
        'coeff': coeff,
        'intercept': intercept,
        'x_mean': x_mean,
        'x_std': x_std,
    }

def predict(model, X):
    # standardize X using the training statistics
    Xs = standardize(X, model['x_mean'], model['x_std'])

    # apply the intercept and coefficients
    y_pred = model['intercept'] + Xs @ model['coeff']
    return y_pred

#### Perform Ridge Regression

In [30]:
# convert CSV strings to numbers
X_train = train_in.astype(float)
y_train = train_out.astype(float)
X_tune = tune_in.astype(float)
y_tune = tune_out.astype(float)

# train, the predict
model = fit(LAMB, X_train, y_train)
y_pred = predict(model, X_tune)

#### Display Ridge Regression Predictions

In [31]:
provinces = ['Metropolitan Manila', 'Nueva Ecija', 'Iloilo', 'South Cotabato', 'Basilan', 'Maguindanao', 'Ifugao', 'Kalinga', 'Ilocos Norte', 'Ilocos Sur', 'Cagayan', 'Isabela',
 'Quirino', 'Bataan', 'Bulacan', 'Batangas', 'Cavite', 'Laguna', 'Quezon', 'Rizal', 'Marinduque', 'Occidental Mindoro', 'Oriental Mindoro', 'Palawan', 'Romblon', 'Zamboanga del Sur',
 'Albay', 'Catanduanes', 'Masbate', 'Sorsogon', 'Aklan', 'Guimaras', 'Negros Occidental', 'Bohol', 'Siquijor', 'Biliran', 'Leyte', 'Northern Samar', 'Samar', 'Southern Leyte', 'Bukidnon',
 'Camiguin', 'Lanao del Norte', 'Misamis Occidental', 'Misamis Oriental', 'Davao Oriental', 'Davao del Norte', 'North Cotabato', 'Sarangani', 'Agusan del Norte', 'Surigao del Norte',
 'Surigao del Sur']

labels = []
with open('../dataset/wfp_food_prices_phl.csv', 'r') as f:
    read = csv.reader(f)
    for x in read:
        if x[0][:4] == '2025' and x[8] == 'Rice (regular, milled)' and x[12] == 'Retail' and x[2] in provinces:
            labels.append({
                'year' : int(x[0][:4]),
                'month' : int(x[0][5:7]),
                'province' : x[2],
                'price' : float(x[14]),
            })

# Stop if the labels do not line up with the 2025 rows
assert len(labels) == len(y_tune)
for i in range(len(labels)):
    assert labels[i]['month'] == X_tune[i, 0]
    assert labels[i]['price'] == y_tune[i]

# Print each prediction next to the correct answer
print(f"{'Month':>5}  {'Province':<22} {'Predicted':>10} {'Actual':>10} {'Error':>8}")
for label, pred, actual in zip(labels, y_pred, y_tune):
    print(f"{label['month']:>5}  {label['province']:<22} {pred:>10.2f} {actual:>10.2f} {pred - actual:>8.2f}")

Month  Province                Predicted     Actual    Error
    1  Basilan                     53.49      50.39     3.10
    1  Maguindanao                 48.28      47.75     0.53
    1  Ifugao                      49.13      48.83     0.30
    1  Kalinga                     50.75      50.88    -0.13
    1  Metropolitan Manila         45.64      44.52     1.12
    1  Ilocos Norte                46.93      43.63     3.30
    1  Ilocos Sur                  46.86      46.23     0.63
    1  Cagayan                     43.88      42.86     1.02
    1  Isabela                     46.69      46.38     0.31
    1  Quirino                     43.31      40.53     2.78
    1  Bataan                      46.72      46.00     0.72
    1  Bulacan                     47.69      44.62     3.07
    1  Bulacan                     47.69      46.73     0.96
    1  Nueva Ecija                 45.02      44.67     0.35
    1  Batangas                    55.39      49.93     5.46
    1  Cavite           

#### Evaluation Metrics on Initial Results

In [32]:
print(f"MAE:  	₱{em.mae(y_tune, y_pred):.4f}/kg")
print(f"RMSE: 	₱{em.rmse(y_tune, y_pred):.4f}/kg")
print(f"MAPE: 	{em.mape(y_tune, y_pred):.4f}%")
print(f"R2:   	{em.r_squared(y_tune, y_pred):.4f}")

MAE:  	₱1.4997/kg
RMSE: 	₱2.0589/kg
MAPE: 	3.5931%
R2:   	0.7923
